# 07 — Gold: FactProductStock (Periodic Snapshot) — Spark SQL

Grain: `(ProductID, SnapshotDateKey)`.

**Técnica Spark SQL:** `INSERT INTO ... SELECT ...` — append-only, imutável por execução.

In [1]:
import sys
import os
sys.path.insert(0, os.getcwd())
from utils import get_spark, register_catalog, WAREHOUSE_DIR
from datetime import date, timedelta

spark = get_spark("NorthwindDW SQL - 07 FactProductStock")
print("Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/29 00:53:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.0


In [2]:
register_catalog(spark)

Catálogo registrado: {'bronze': 11, 'silver': 0, 'gold': 11}


In [3]:
today    = date.today()
snap_key = int(today.strftime("%Y%m%d"))
print(f"SnapshotDateKey: {snap_key}")

existing = spark.sql(f"SELECT COUNT(*) AS n FROM gold.FactProductStock WHERE SnapshotDateKey = {snap_key}").collect()[0]["n"]
if existing > 0:
    print(f"Snapshot de hoje ({snap_key}) já existe ({existing} linhas). Nenhuma ação.")
else:
    spark.sql(f"""
        INSERT INTO gold.FactProductStock
        SELECT
            ABS(HASH({snap_key}, dp.ProductSK)) AS StockSK,
            {snap_key}                           AS SnapshotDateKey,
            dp.ProductSK,
            dc.CategorySK,
            CAST(p.UnitsInStock  AS INT),
            CAST(p.UnitsOnOrder  AS INT),
            CAST(p.ReorderLevel  AS INT),
            (p.UnitsInStock <= p.ReorderLevel)   AS NeedsReorder,
            current_timestamp()                  AS LoadTimestamp
        FROM bronze.products p
        JOIN gold.DimProduct  dp ON p.ProductID     = dp.ProductID   AND dp.IsCurrent = true
        JOIN gold.DimCategory dc ON dp.CategoryName = dc.CategoryName
    """)
    n = spark.sql(f"SELECT COUNT(*) AS n FROM gold.FactProductStock WHERE SnapshotDateKey = {snap_key}").collect()[0]["n"]
    print(f"Snapshot inserido: {snap_key} | {n} produtos")

SnapshotDateKey: 20260329


Snapshot inserido: 20260329 | 77 produtos


In [4]:
print("1. Snapshots disponíveis:")
spark.sql("""
    SELECT SnapshotDateKey, COUNT(*) AS count FROM gold.FactProductStock
    GROUP BY SnapshotDateKey ORDER BY SnapshotDateKey
""").show()

max_snap = spark.sql("SELECT MAX(SnapshotDateKey) AS m FROM gold.FactProductStock").collect()[0]["m"]
if max_snap:
    print(f"2. Produtos com NeedsReorder (último snapshot: {max_snap}):")
    spark.sql(f"""
        SELECT dp.ProductName, dp.CategoryName,
               fs.UnitsInStock, fs.ReorderLevel, fs.UnitsOnOrder
        FROM gold.FactProductStock fs
        JOIN gold.DimProduct dp ON fs.ProductSK = dp.ProductSK AND dp.IsCurrent = true
        WHERE fs.NeedsReorder = true AND fs.SnapshotDateKey = {max_snap}
        ORDER BY UnitsInStock - ReorderLevel
    """).show(truncate=False)

dups = spark.sql("""
    SELECT COUNT(*) AS n FROM (
        SELECT SnapshotDateKey, ProductSK FROM gold.FactProductStock
        GROUP BY SnapshotDateKey, ProductSK HAVING COUNT(*) > 1
    )
""").collect()[0]["n"]
print(f"3. Grain único: {dups} duplicatas (esperado: 0)")

1. Snapshots disponíveis:


+---------------+-----+
|SnapshotDateKey|count|
+---------------+-----+
|       20260329|   77|
+---------------+-----+



2. Produtos com NeedsReorder (último snapshot: 20260329):


+-------------------------+--------------+------------+------------+------------+
|ProductName              |CategoryName  |UnitsInStock|ReorderLevel|UnitsOnOrder|
+-------------------------+--------------+------------+------------+------------+
|Gorgonzola Telino        |Dairy Products|0           |20          |70          |
|Mascarpone Fabioli       |Dairy Products|9           |25          |40          |
|Louisiana Hot Spiced Okra|Condiments    |4           |20          |100         |
|Outback Lager            |Beverages     |15          |30          |10          |
|Gravad lax               |Seafood       |11          |25          |50          |
|Aniseed Syrup            |Condiments    |13          |25          |70          |
|Rogede sild              |Seafood       |5           |15          |70          |
|Chocolade                |Confections   |15          |25          |70          |
|Gnocchi di nonna Alice   |Grains/Cereals|21          |30          |10          |
|Scottish Longbr

3. Grain único: 0 duplicatas (esperado: 0)


In [5]:
# Simular snapshot de ontem para demonstração
ontem = int((date.today() - timedelta(days=1)).strftime("%Y%m%d"))
already = spark.sql(f"SELECT COUNT(*) AS n FROM gold.FactProductStock WHERE SnapshotDateKey = {ontem}").collect()[0]["n"]
if already == 0:
    spark.sql(f"""
        INSERT INTO gold.FactProductStock
        SELECT ABS(HASH({ontem}, ProductSK)) AS StockSK,
               {ontem} AS SnapshotDateKey,
               ProductSK, CategorySK, UnitsInStock, UnitsOnOrder,
               ReorderLevel, NeedsReorder, current_timestamp() AS LoadTimestamp
        FROM gold.FactProductStock WHERE SnapshotDateKey = {snap_key}
    """)
    n = spark.sql(f"SELECT COUNT(*) AS n FROM gold.FactProductStock WHERE SnapshotDateKey = {ontem}").collect()[0]["n"]
    print(f"Snapshot simulado: {ontem} ({n} produtos)")
else:
    print(f"Snapshot de ontem ({ontem}) já existe.")

Snapshot simulado: 20260328 (77 produtos)


In [6]:
print("Histórico de versões Delta:")
spark.sql("DESCRIBE HISTORY gold.FactProductStock").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(10, truncate=False)

Histórico de versões Delta:


+-------+-----------------------+------------+------------------------------------------------------------+
|version|timestamp              |operation   |operationMetrics                                            |
+-------+-----------------------+------------+------------------------------------------------------------+
|2      |2026-03-29 00:54:21.647|WRITE       |{numFiles -> 1, numOutputRows -> 77, numOutputBytes -> 3769}|
|1      |2026-03-29 00:54:10.203|WRITE       |{numFiles -> 1, numOutputRows -> 77, numOutputBytes -> 3769}|
|0      |2026-03-29 00:42:47.774|CREATE TABLE|{}                                                          |
+-------+-----------------------+------------+------------------------------------------------------------+

